# Øvelse: Barometerets brugerflade

I dag får barometeret tre nye ting: **en figur**, **en besked**, når sammenligningen
måtte gøres bredere, og **en opslagsside**, hvor man kan se udviklingen fra 2012 til
2022 uden at taste noget ind.

### Det vigtigste at vide først

Koden ligger **allerede færdig** i `app/barometer/figur.py`, `tekst.py` og `opslag.py`.
Appen virker, også før I går i gang. Den her notebook er stedet, vi prøver at forstå den,
linje for linje.

Derfor står koden her igen, men med huller. Hvert trin har en længere forklaring, en
kodecelle med `_______` de steder, I skal udfylde, og en **tjek-celle**, der siger, om
det virkede. De sidste tjek-celler i hver del sammenligner jeres udgave med filen i
appen.

### Praktisk

Notebooken skal ligge i **roden** af repositoriet, ved siden af `app/` og `data/`.
Kør cellerne oppefra og ned med `Shift + Enter`. Opfører noget sig mærkeligt, så prøv først
*Restart* og *Run All*.

## 0: Gør klar

Først henter vi det, I byggede de sidste to gange: klassen `Arbejdsplads` og
referencetallene. Vi laver to arbejdspladser, som vi bruger hele vejen igennem:

- **Skolen**: 120 ansatte, offentlig. Den findes der præcise sammenligningstal for.
- **Værket**: 60 ansatte, privat industri. Den gør der *ikke*. Den skal vise os,
  hvad der sker, når sammenligningen må gøres bredere.

`import` henter kode fra andre filer. `matplotlib.pyplot` tegner figurer, og `pandas`
arbejder med tabeller. De to er ikke en del af Python selv. D

Der er ingen huller her. Kør cellen, og så tjek-cellen.

In [1]:
import matplotlib.pyplot as plt
import pandas as pd

from app.barometer import Arbejdsplads, Referencetal
from app.barometer.referencetal import DIMENSIONER

referencetal = Referencetal()

skolen = Arbejdsplads(120, {"kvinder": 84, "seniorer": 26, "indvandrere": 9},
                      "Undervisning", "Offentlig", "Nordjylland")
vaerket = Arbejdsplads(60, {"kvinder": 15, "seniorer": 12, "indvandrere": 8},
                       "Industri", "Privat", "Nordjylland")

print(skolen)
print(vaerket)

ModuleNotFoundError: No module named 'app'

In [ ]:
# tjek
if skolen.andel("kvinder") == 0.7 and skolen.stoerrelsesgruppe == "100-249":
    print("Klar. Arbejdsplads virker, og referencetallene er hentet.")
else:
    print("Arbejdsplads er ikke færdig fra sidste gang. Gør trin 14 i sidste uges notebook,")
    print("og kør python3 tests/test_arbejdsplads.py, før I går videre.")

# Del 1 · Figuren

## 1 · Lærred og tegneflade

En figur i matplotlib består af to ting:

- **Figuren** (`fig`) er hele billedet — lærredet.
- **Akserne** (`ax`) er én tegneflade på lærredet, med en x-akse og en y-akse. En figur
  kan have flere tegneflader ved siden af hinanden. Det får vi brug for om lidt.

`plt.subplots()` laver begge dele på én gang og giver dem tilbage som to værdier. Det
kender I fra sidste uge: en funktion kan returnere to ting, adskilt af et komma.

`ax.barh()` tegner en vandret bjælke. Den skal have tre ting at vide:

| | Betyder |
|---|---|
| det første tal | hvor højt oppe bjælken står — her bare 0 |
| det andet tal | hvor bred den er |
| `left=` | hvor den **starter** |

**Udfyld hullet:** bjælken skal starte ved 20 procent, altså `0.2`, og være 50 procent
bred.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 1.5))

ax.barh(0, 0.5, left=_______, height=0.5, color="#8fb0e3")

ax.set_xlim(0, 1)        # x-aksen går fra 0 til 1, altså fra 0 til 100 procent
ax.set_yticks([])        # ingen tal på y-aksen

In [ ]:
# tjek
bjaelke = ax.patches[0]

if abs(bjaelke.get_x() - 0.2) < 1e-9 and abs(bjaelke.get_width() - 0.5) < 1e-9:
    print("Rigtigt. Bjælken starter ved 20 procent og slutter ved 70.")
else:
    print(f"Ikke endnu. Bjælken starter ved {bjaelke.get_x()}. Den skal starte ved 0.2.")

## 2 · Ét panel

Nu tegner vi ét af barometerets tre paneler. Det fortæller, hvor arbejdspladsen ligger
blandt de sammenlignelige — og det gør det med fire lag oven på hinanden.

Først skal vi huske, hvad tallene betyder. For hver sammenligningsgruppe har vi fem
**percentiler**:

| | Betyder |
|---|---|
| `p10` | 10 procent af arbejdspladserne ligger under det tal |
| `p25` | en fjerdedel ligger under |
| `p50` | **medianen**: halvdelen ligger under, halvdelen over |
| `p75` | tre fjerdedele ligger under |
| `p90` | 90 procent ligger under |

De fire lag i panelet er:

1. **Det brede, lyse bånd** fra `p10` til `p90`. Her ligger de midterste 80 procent af
   arbejdspladserne.
2. **Det smalle, mørkere bånd** fra `p25` til `p75`. Her ligger den midterste halvdel.
3. **En lodret streg** ved medianen.
4. **En rød trekant** ved arbejdspladsens egen andel.

Bjælkens bredde regnes ud som *slut minus start*: `p90 - p10`.

Funktionen `tegn_panel` får tegnefladen `ax` med ind og tegner på den. Den returnerer
ikke noget — den ændrer på det objekt, den fik. Det kalder Downey en **modifier**
(kapitel 14).

**Udfyld hullerne:** hvor starter det brede bånd? Og hvilken af parametrene er
arbejdspladsens egen andel?

In [ ]:
NAVNE = {"kvinder": "Kvinder", "seniorer": "Seniorer (55+)", "indvandrere": "Indvandrerbaggrund"}

LYS = "#dbe6f6"
MELLEM = "#8fb0e3"
MOERK = "#1f5fbf"
ROED = "#c0392b"


def tegn_panel(ax, navn, egen_andel, raekke):
    """Tegn ét panel: fordelingen i sammenligningsgruppen og arbejdspladsens egen andel."""
    # 1 · det brede bånd: fra p10 til p90
    ax.barh(0, raekke["p90"] - raekke["p10"], left=raekke["_______"], height=0.5, color=LYS)

    # 2 · det smalle bånd: fra p25 til p75
    ax.barh(0, raekke["p75"] - raekke["p25"], left=raekke["p25"], height=0.5, color=MELLEM)

    # 3 · medianen som en lodret streg
    ax.plot([raekke["p50"], raekke["p50"]], [-0.25, 0.25], color=MOERK, linewidth=2)

    # 4 · arbejdspladsen selv: en rød trekant
    ax.plot(_______, 0, marker="v", markersize=12, color=ROED)

    ax.set_title(NAVNE[navn], fontsize=10)
    ax.set_xlim(0, 1)
    ax.set_yticks([])
    ax.xaxis.set_major_formatter(lambda x, _: f"{x:.0%}")


# Prøv den: skolens andel af kvinder
raekke, beskrivelse = referencetal.find(skolen, 1)
print("Sammenlignet med:", beskrivelse)

fig, ax = plt.subplots(figsize=(4, 2))
tegn_panel(ax, "kvinder", skolen.andel("kvinder"), raekke)

In [ ]:
# tjek
try:
    _fig, _ax = plt.subplots()
    tegn_panel(_ax, "kvinder", skolen.andel("kvinder"), raekke)
    bred_ok = abs(_ax.patches[0].get_x() - raekke["p10"]) < 1e-9
    trekant_ok = abs(_ax.lines[-1].get_xdata()[0] - skolen.andel("kvinder")) < 1e-9
    plt.close("all")
    if bred_ok and trekant_ok:
        print("Rigtigt. Det brede bånd starter ved p10, og trekanten står ved skolens egen andel.")
    elif not bred_ok:
        print("Ikke endnu. Det brede bånd skal starte ved p10.")
    else:
        print("Ikke endnu. Trekanten skal stå ved arbejdspladsens egen andel.")
except Exception as fejl:
    plt.close("all")
    print("Ikke endnu. Python sagde:", type(fejl).__name__, fejl)

## 3 · Tre paneler

Barometeret viser altid **tre** paneler — ét for kvinder, ét for seniorer og ét for
indvandrerbaggrund. Aldrig ét samlet tal. Det står i køreplanen, og her er det, det
bliver til kode.

`plt.subplots(1, 3)` laver én figur med **tre** tegneflader ved siden af hinanden:
én række, tre kolonner. `akser` er nu en samling af tre `ax`.

Så skal hver tegneflade have sin dimension. Det gør `zip`. Den sætter to samlinger
sammen to og to — som en lynlås. Første tegneflade med første dimension, anden med
anden, og så videre. I møder `zip` i Think Python kapitel 11 om tupler; her skal I
bare bruge den.

`DIMENSIONER.items()` giver par af `(1, "kvinder")`, `(2, "seniorer")` og
`(3, "indvandrere")`. Derfor står der `(dimension_id, navn)` i parentes: hvert par
pakkes ud i to navne.

Til sidst **returnerer** funktionen figuren. Den viser den ikke selv. Det er appen, der
bestemmer, hvor figuren skal vises. Det er lagene fra lektion 4 igen: kernen tegner,
appen viser.

**Udfyld hullerne:** hvilken metode på `Arbejdsplads` giver en andel? Og hvad skal
funktionen give tilbage?

In [ ]:
def tegn(arbejdsplads, referencetal):
    """Tegn alle tre paneler for én arbejdsplads. Returnerer figuren."""
    fig, akser = plt.subplots(1, 3, figsize=(10, 2.6))

    for ax, (dimension_id, navn) in zip(akser, DIMENSIONER.items()):
        raekke, _ = referencetal.find(arbejdsplads, dimension_id)
        tegn_panel(ax, navn, arbejdsplads._______(navn), raekke)

    fig.tight_layout()
    return _______


figur = tegn(skolen, referencetal)

In [ ]:
# tjek
from app.barometer.figur import tegn as filens_tegn

try:
    jeres, filens = tegn(skolen, referencetal), filens_tegn(skolen, referencetal)
    ens = len(jeres.axes) == len(filens.axes) == 3 and all(
        abs(a.lines[-1].get_xdata()[0] - b.lines[-1].get_xdata()[0]) < 1e-9
        and abs(a.patches[0].get_x() - b.patches[0].get_x()) < 1e-9
        for a, b in zip(jeres.axes, filens.axes))
    plt.close("all")
    print("Rigtigt. Jeres tegn() giver præcis den samme figur som app/barometer/figur.py."
          if ens else "Ikke endnu. Figuren ligner ikke den i appen.")
except Exception as fejl:
    plt.close("all")
    print("Ikke endnu. Python sagde:", type(fejl).__name__, fejl)

# Del 2 · Beskeden

## 4 · Står teksten der?

Når en sammenligningsgruppe mangler i tallene, finder `Referencetal.find` en bredere —
og skriver det i beskrivelsen. For værket ser beskrivelsen sådan ud:

```
Industri, 50-99 ansatte, begge sektorer
```

Barometeret skal opdage det og sige det til brugeren. Til det bruger vi `in`.
`"abc" in "xabcx"` er `True`, fordi teksten til venstre står et sted i teksten til
højre. Det er den samme `in`, som I kender fra `for`-løkker, bare brugt på tekst. I
møder den i Think Python kapitel 8 om strenge.

Læg mærke til, at der skal stå **præcis** det samme: store og små bogstaver tæller.
`"Begge sektorer"` står ikke i beskrivelsen ovenfor.

**Udfyld hullet:** hvilke to ord står i beskrivelsen, når sektoren blev gjort bredere?

In [ ]:
beskrivelse = "Industri, 50-99 ansatte, begge sektorer"

med_sektor = "_______" in beskrivelse
med_privat = "privat sektor" in beskrivelse

print(med_sektor)
print(med_privat)

In [ ]:
# tjek
if med_sektor is True and med_privat is False:
    print("Rigtigt. 'begge sektorer' står i beskrivelsen, 'privat sektor' gør ikke.")
else:
    print("Ikke endnu. med_sektor skal være True. Står ordene præcis som i beskrivelsen?")

## 5 · Beskeden

Nu bliver det til en funktion. Den skal gøre én af to ting:

- Er gruppen blevet **bredere** — sektoren eller størrelsen er der set bort fra — så
  returnér en besked, brugeren kan læse.
- Ellers returnér **ingenting**.

I Python hedder "ingenting" `None`. Think Python kapitel 6 har et afsnit, der hedder
*And some have None*: en funktion uden `return` giver `None` af sig selv. Her skriver vi
det udtrykkeligt, så den, der læser koden, kan se, at det er meningen.

`or` betyder, at det er nok, at **én** af de to ting er sand. Gruppen kan være gjort
bredere på to måder: begge sektorer, eller alle størrelser.

**Udfyld hullerne.**

In [ ]:
def bredere_besked(beskrivelse):
    """Returnér en besked, hvis gruppen blev gjort bredere. Ellers None."""
    if "_______" in beskrivelse or "alle størrelser" in beskrivelse:
        return ("Der var ikke tal nok til den præcise sammenligning. "
                f"I stedet sammenlignes med: {beskrivelse}.")
    return _______


print(bredere_besked("Industri, 50-99 ansatte, begge sektorer"))
print(bredere_besked("Handel, alle størrelser"))
print(bredere_besked("Undervisning, 100-249 ansatte, offentlig sektor"))

In [ ]:
# tjek
from app.barometer.tekst import bredere_besked as filens_besked

proever = ["Industri, 50-99 ansatte, begge sektorer",
           "Handel, alle størrelser",
           "Undervisning, 100-249 ansatte, offentlig sektor"]
try:
    if all(bredere_besked(b) == filens_besked(b) for b in proever):
        print("Rigtigt. Jeres bredere_besked() svarer præcis som den i app/barometer/tekst.py.")
    else:
        print("Ikke endnu. Den præcise gruppe skal give None, de to andre en besked.")
except Exception as fejl:
    print("Ikke endnu. Python sagde:", type(fejl).__name__, fejl)

## 6 · Alle beskeder, kun én gang

En arbejdsplads har tre dimensioner, så `find` bliver kaldt tre gange. For værket
mangler gruppen i alle tre — og så ville brugeren få den samme besked tre gange.

Det løser vi med en **liste**:

- `ud = []` laver en tom liste.
- `ud.append(besked)` lægger en besked bagest i listen.
- `besked not in ud` er `True`, hvis beskeden **ikke** allerede står i listen.

Så kommer hver besked kun med én gang. Lister har I ikke læst om endnu — det er Think
Python kapitel 9 — men det her er alt, I skal bruge i dag.

`_` i `_, beskrivelse = ...` er et navn som alle andre. Man bruger det, når man får to
ting tilbage og kun skal bruge den ene.

**Udfyld hullet:** hvilken metode lægger noget i en liste?

In [ ]:
def beskeder(arbejdsplads, referencetal):
    """Alle beskeder for én arbejdsplads — hver besked kun én gang."""
    ud = []
    for dimension_id in DIMENSIONER:
        _, beskrivelse = referencetal.find(arbejdsplads, dimension_id)
        besked = bredere_besked(beskrivelse)
        if besked is not None and besked not in ud:
            ud._______(besked)
    return ud


print("Værket:", beskeder(vaerket, referencetal))
print("Skolen:", beskeder(skolen, referencetal))

In [ ]:
# tjek
try:
    v, s = beskeder(vaerket, referencetal), beskeder(skolen, referencetal)
    if len(v) == 1 and s == []:
        print("Rigtigt. Værket får én besked, skolen ingen.")
    else:
        print(f"Ikke endnu. Værket fik {len(v)} beskeder og skolen {len(s)}. Det skal være 1 og 0.")
except Exception as fejl:
    print("Ikke endnu. Python sagde:", type(fejl).__name__, fejl)

# Del 3 · Opslaget

## 7 · Vælg rækkerne

Opslagssiden skal vise **medianen for hver branche** i 2012 og 2022 — for en hel
branche, uden at dele den op i størrelser eller sektorer.

`referencetal.tal` er den store tabel, som I satte sammen med `merge` sidste gang. Vi
skal have de rækker, hvor:

- dimensionen er den, vi kigger på — her `1`, kvinder
- branchen **ikke** er `"Alle"` — vi vil have hver branche for sig
- størrelsen er `"Alle"` — hele branchen, ikke kun de små
- sektoren er `"Alle"` — begge sektorer

Det er den samme måde at vælge rækker på som i trin 10 sidste uge: hver betingelse i sin
egen parentes, og `&` imellem. `!=` betyder *er ikke lig med*.

Der er 8 brancher og 2 år, så der skal komme 16 rækker.

**Udfyld hullet:** hvilken branche skal ikke med på en liste over brancher?

In [ ]:
tal = referencetal.tal

udvalg = tal[(tal["dimension_id"] == 1)
             & (tal["branche"] != "_______")
             & (tal["stoerrelse"] == "Alle")
             & (tal["sektor"] == "Alle")]

print(len(udvalg), "rækker")
print(udvalg[["aar", "branche", "p50"]].head(4))

In [ ]:
# tjek
if len(udvalg) == 16:
    print("Rigtigt. 8 brancher gange 2 år.")
elif len(udvalg) == 28:
    print("Ikke endnu. Der er 12 rækker for meget: 'Alle' er ikke sorteret fra.")
    print("Der er seks Alle-rækker pr. år — hele landet og én pr. region.")
else:
    print(f"Ikke endnu. Der kom {len(udvalg)} rækker. Der skal være 16.")

## 8 · Fra lang til bred

Tabellen fra trin 7 er **lang**: én række pr. branche *og* år. Handel står der to gange
— én gang for 2012 og én gang for 2022.

```
aar   branche   p50
2012  Handel    0.409
2022  Handel    0.454
2012  Industri  0.266
2022  Industri  0.295
```

Til opslagssiden vil vi have den **bred**: én række pr. branche, og én kolonne pr. år.

```
branche   2012   2022
Handel    0.409  0.454
Industri  0.266  0.295
```

Det gør `pivot`. Den skal vide tre ting:

| | Betyder |
|---|---|
| `index=` | hvad der skal stå ned ad rækkerne — branchen |
| `columns=` | hvad der skal blive til kolonner — året |
| `values=` | hvilket tal der skal stå i cellerne — medianen |

**Udfyld hullet:** hvilken kolonne er medianen?

In [ ]:
tabel = udvalg.pivot(index="branche", columns="aar", values="_______")

print(tabel)

In [ ]:
# tjek
try:
    if tabel.shape == (8, 2) and abs(tabel.loc["Handel", 2022] -
           udvalg[(udvalg["branche"] == "Handel") & (udvalg["aar"] == 2022)]["p50"].iloc[0]) < 1e-9:
        print("Rigtigt. Otte brancher, to år, og medianen i cellerne.")
    else:
        print("Ikke endnu. Tabellen skal have 8 rækker, 2 kolonner, og medianen — p50 — i cellerne.")
except Exception as fejl:
    print("Ikke endnu. Python sagde:", type(fejl).__name__, fejl)

## 9 · Ændringen og pæne navne

Nu gør vi tabellen klar til at vise frem. Hver linje gør én ting:

1. `* 100` — fra andele (0.454) til procent (45.4). Det er nemmere at læse.
2. En ny kolonne med **ændringen**: 2022 minus 2012. Er tallet positivt, er andelen
   steget.
3. `round(1)` — én decimal er nok.
4. `reset_index()` — branchen var blevet til rækkernes navne under `pivot`. Nu bliver den
   en almindelig kolonne igen.
5. Nye kolonnenavne, som et menneske kan læse.
6. `sort_values` — højeste median øverst. `ascending=False` betyder *ikke stigende*,
   altså faldende.

Læg mærke til, at årstallene er **tal** i kolonnenavnene, ikke tekst: der står `2022`,
ikke `"2022"`. De kom fra kolonnen `aar`, og der var de tal.

**Udfyld hullerne:** hvilket år trækkes fra hvilket? Og skal den sortere stigende eller
faldende?

In [ ]:
tabel = udvalg.pivot(index="branche", columns="aar", values="p50") * 100
tabel["Ændring"] = tabel[_______] - tabel[2012]

tabel = tabel.round(1).reset_index()
tabel.columns = ["Branche", "Median 2012 (%)", "Median 2022 (%)", "Ændring (procentpoint)"]
tabel = tabel.sort_values("Median 2022 (%)", ascending=_______)

tabel

In [ ]:
# tjek
try:
    forskel = (tabel["Median 2022 (%)"] - tabel["Median 2012 (%)"] - tabel["Ændring (procentpoint)"]).abs().max()
    if forskel < 0.11 and tabel["Median 2022 (%)"].is_monotonic_decreasing:
        print("Rigtigt. Ændringen er 2022 minus 2012, og den højeste median står øverst.")
    elif forskel >= 0.11:
        print("Ikke endnu. Ændringen skal være 2022 minus 2012.")
    else:
        print("Ikke endnu. Den højeste median skal stå øverst — faldende rækkefølge.")
except Exception as fejl:
    print("Ikke endnu. Python sagde:", type(fejl).__name__, fejl)

## 10 · Saml det i én funktion

Trin 7, 8 og 9 var tre stykker kode, der kun virkede for kvinder, fordi der stod `1`.
Nu samler vi dem i én funktion, der virker for alle tre dimensioner.

Det er Downeys udviklingsplan fra lektion 4: først fik vi det til at virke for ét
tilfælde, så **indkapsler** vi det i en funktion, og så **generaliserer** vi ved at gøre
dimensionen til en parameter.

`DIMENSION_ID` oversætter fra navn til nummer, så man kan skrive `"kvinder"` i stedet for
at huske, at det er `1`.

Der er ingen huller. Kør cellen, og se tabellen for seniorer. Tjek-cellen sammenligner
med filen i appen — for alle tre dimensioner.

In [ ]:
DIMENSION_ID = {"kvinder": 1, "seniorer": 2, "indvandrere": 3}


def brancher(referencetal, dimension):
    """Medianen for hver branche i 2012 og 2022 — og ændringen. Returnerer en tabel."""
    tal = referencetal.tal

    udvalg = tal[(tal["dimension_id"] == DIMENSION_ID[dimension])
                 & (tal["branche"] != "Alle")
                 & (tal["stoerrelse"] == "Alle")
                 & (tal["sektor"] == "Alle")]

    tabel = udvalg.pivot(index="branche", columns="aar", values="p50") * 100
    tabel["Ændring"] = tabel[2022] - tabel[2012]

    tabel = tabel.round(1).reset_index()
    tabel.columns = ["Branche", "Median 2012 (%)", "Median 2022 (%)", "Ændring (procentpoint)"]
    return tabel.sort_values("Median 2022 (%)", ascending=False)


brancher(referencetal, "seniorer")

In [ ]:
# tjek
from app.barometer.opslag import brancher as filens_brancher

ens = all(brancher(referencetal, d).reset_index(drop=True)
          .equals(filens_brancher(referencetal, d).reset_index(drop=True))
          for d in ["kvinder", "seniorer", "indvandrere"])
print("Rigtigt. Jeres brancher() giver præcis den samme tabel som app/barometer/opslag.py."
      if ens else "Ikke endnu. Tabellen ligner ikke den i appen.")

# Del 4 · I appen

## 11 · Start appen

Alt det, I lige har skrevet, står i filerne i `app/barometer/`. Appen bruger dem.
Først et sidste tjek af filerne — kør cellen nedenfor.

Åbn derefter en **terminal** i VS Code (*Terminal → New Terminal*), sørg for, at I står
i roden af repositoriet, og skriv:

```
python -m shiny run --reload app/app.py
```

Åbn http://127.0.0.1:8000 i browseren. Prøv:

- **Barometer:** skift til Industri, Privat og 60 ansatte. Der skal komme en blå besked
  øverst — det er jeres `bredere_besked`.
- **Opslag:** skift mellem de tre dimensioner. Tabellen er jeres `brancher`.
- **Om:** den er næsten tom. Den skriver I i trin 12.

`--reload` betyder, at appen starter forfra af sig selv, hver gang I gemmer en fil. Lad
den køre.

In [ ]:
import subprocess, sys

resultat = subprocess.run([sys.executable, "tests/test_udvidelser.py"],
                          capture_output=True, text=True)
print(resultat.stdout[-600:])

## 12 · Om-siden

Åbn `app/om.md`. Det er den side, der står under fanen *Om* — og den er den vigtigste
side i appen, fordi den er grunden til, at nogen kan stole på de to andre.

Skriv den for en, der aldrig har hørt om projektet. **Ingen kode.** Fire afsnit:

1. **Hvad barometeret kan**, og hvem det er til.
2. **Hvor tallene kommer fra** — og hvorfor de er syntetiske lige nu.
3. **Hvad barometeret ikke kan.** Brug de fem beslutninger fra `KOEREPLAN.md` — mindst
   tre af dem, forklaret med jeres egne ord.
4. **Hvem der har lavet det.**

Slet kommentarerne mellem `<!--` og `-->`, når I er færdige. Gem filen, og klik på
fanen *Om* i appen. Den opdaterer sig selv.

Til sidst: **commit og push.** Og bed en anden i gruppen om at pulle og starte appen på
sin egen maskine. Kører det kun hos den, der skrev det, virker det ikke.

In [ ]:
# Sådan ser Om-siden ud lige nu. Kør cellen igen, når I har skrevet den.
print(open("app/om.md", encoding="utf-8").read())